# Benchmark results and figures

This notebook loads saved rare-cell downsampling benchmark results and generates publication-style figures for rare-cell recovery, neighborhood preservation, and representation comparison.

**Required input:** `results/tables/{output_prefix}__benchmark_results.csv`

**Outputs saved to:** `results/figures/` (PNG and PDF), `results/tables/{output_prefix}__figure_index.csv`


## Setup

Run this notebook after `rare_cell_downsampling_benchmark.ipynb` has saved result tables.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml as _yaml
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from rarecell.benchmark import TABLES_DIR
from rarecell.plotting import save_all_standard_plots
from rarecell.plotting import set_plot_style
from rarecell.utils import resolve_existing_output_prefix

set_plot_style()

# ── Parameters ────────────────────────────────────────────────────────────────
# Read optional config to detect the expected output prefix, then auto-discover.
_config_path = PROJECT_ROOT / "config" / "benchmark_config.yaml"
_config: dict = {}
if _config_path.exists():
    _config = _yaml.safe_load(_config_path.read_text()) or {}

# preferred_output_prefix: set explicitly (e.g. "pbmc5k_14") to pin a specific
# run; leave as None to auto-detect the most recently written result files.
preferred_output_prefix: str | None = None

output_prefix = resolve_existing_output_prefix(TABLES_DIR, preferred_output_prefix)
results_path = TABLES_DIR / f"{output_prefix}__benchmark_results.csv"
summary_path = TABLES_DIR / f"{output_prefix}__metric_summary.csv"
print(f"Using output prefix: {output_prefix!r}")

## Load benchmark results

The primary result table is produced by `rare_cell_downsampling_benchmark.ipynb`.

In [ ]:
results = pd.read_csv(results_path)
print(f"Loaded {len(results)} rows, {results['representation'].nunique()} representations, "
      f"{results['retain_fraction'].nunique()} fractions")
display(results.head())

summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
if not summary.empty:
    display(summary.head())

## Validate the result table

Checks that the required metric columns are present before generating figures.


In [ ]:
required_columns = [
    "target_label", "retain_fraction", "seed", "representation",
    "n_target", "precision", "recall", "f1",
    "neighborhood_purity", "target_silhouette",
]
missing = [c for c in required_columns if c not in results.columns]
if missing:
    raise ValueError(f"Benchmark results are missing required columns: {missing}")
if results.empty:
    raise ValueError("Benchmark results table is empty.")
print(f"Loaded {len(results)} rows, {results['representation'].nunique()} representations, "
      f"{results['retain_fraction'].nunique()} fractions")


## Generate benchmark figures

All standard figures are generated by `save_all_standard_plots()` from `src/rarecell/benchmark_plots.py`. Each figure is saved as both PNG and PDF. Paths are recorded in the figure index CSV.


In [ ]:
plot_paths = save_all_standard_plots(results, output_prefix)
print(f"Saved {len(plot_paths)} figure files (PNG + PDF).")
for path in sorted(set(plot_paths)):
    print(f"  {path}")


## Display figures


In [ ]:
for path in plot_paths:
    if str(path).endswith(".png"):
        print(path.name)
        display(Image(filename=str(path)))


## Metric summary table

The summary shows mean metric values across seeds, grouped by representation and retained fraction.


In [ ]:
if not summary.empty:
    display(summary)
else:
    from rarecell.benchmark import make_metric_summary

    summary = make_metric_summary(results)
    display(summary)


## Auto-interpretation

This short summary identifies the best representation at the lowest retained fraction.


In [ ]:
lowest_fraction = float(results["retain_fraction"].min())
metric_cols = [c for c in ["f1", "recall", "neighborhood_purity"] if c in results.columns]
mean_by_rep = (
    results[results["retain_fraction"] == lowest_fraction]
    .groupby("representation")[metric_cols]
    .mean(numeric_only=True)
    .reset_index()
)
if not mean_by_rep.empty and "f1" in mean_by_rep.columns:
    best = mean_by_rep.sort_values("f1", ascending=False).iloc[0]
    print(f"At lowest fraction ({lowest_fraction:.0%} retained target cells):")
    print(f"  Best F1:  {best['representation']} ({best['f1']:.3f})")
display(mean_by_rep.round(3))


## Figure index

The figure index CSV records all generated figure paths and is written alongside the figures.


In [ ]:
figure_index_path = TABLES_DIR / f"{output_prefix}__figure_index.csv"
if figure_index_path.exists():
    display(pd.read_csv(figure_index_path))


## Summary

In [ ]:
# Summary of generated outputs and deviations from run_downsampling_benchmark.py
print("Generated outputs:")
for p in sorted(set(plot_paths)):
    status = "OK" if p.exists() else "MISSING"
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = p
    print(f"  [{status}] {rel}")

figure_index_path = TABLES_DIR / f"{output_prefix}__figure_index.csv"
if figure_index_path.exists():
    try:
        rel = figure_index_path.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = figure_index_path
    print(f"  [OK] {rel}")

print()
print("Deviations from run_downsampling_benchmark.py:")
print("  - This notebook only generates figures from pre-saved benchmark results.")
print("  - The benchmark itself is run by rare_cell_downsampling_benchmark.ipynb.")
